# Solution C1 · Temporal leakage: random K-fold on a time series

**Type**: Break the model

## Objective

Show that on a time series a random K-fold gives an R² close to 1 for a model that actually cannot predict the
future, compare it with `TimeSeriesSplit` and with a true out-of-sample forecast, and compare the model with a seasonal baseline.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score

s = ds.time_series()
X, y = s[["t", "s", "c"]], s["y"]
rf = RandomForestRegressor(100, random_state=0)
cut = 480

## Commented solution

In [ ]:
r2_kfold = cross_val_score(rf, X, y, cv=KFold(5, shuffle=True, random_state=0), scoring="r2").mean()
r2_ts = cross_val_score(rf, X, y, cv=TimeSeriesSplit(5), scoring="r2").mean()
print(f"R2 with random KFold: {r2_kfold:.3f} | with TimeSeriesSplit: {r2_ts:.3f}")

In [ ]:
rf.fit(X.iloc[:cut], y.iloc[:cut])
r2_future = r2_score(y.iloc[cut:], rf.predict(X.iloc[cut:]))
r2_seasonal = r2_score(y.iloc[cut:], y.shift(12).iloc[cut:])
kfold_distance = abs(r2_kfold - r2_future)
ts_distance = abs(r2_ts - r2_future)
print(f"true R2 on the last 120 months: {r2_future:.3f} | forecast 'same month of the previous year': {r2_seasonal:.3f}")
print(f"error of the estimate: KFold {kfold_distance:.3f} | TimeSeriesSplit {ts_distance:.3f}")

## Check

In [ ]:
with verify():
    check(r2_kfold > 0.9, "the random K-fold gives a very high R2 (above 0.9)", "with random KFold the model interpolates between adjacent months")
    check(r2_ts < 0.6, "with TimeSeriesSplit the R2 collapses (below 0.6)", "TimeSeriesSplit trains on the past and evaluates on the future")
    check(close(kfold_distance, abs(r2_kfold - r2_future)) and close(ts_distance, abs(r2_ts - r2_future)), "distances computed correctly",
          "distance = |estimated R2 - true R2 on the future|")
    check(ts_distance < kfold_distance / 2, "the temporal estimate is much closer to the truth than the random K-fold one",
          "TimeSeriesSplit must get much closer to the true R2 on the future than the random KFold")
    check(r2_seasonal > r2_future + 0.3, "the baseline 'same month of the previous year' beats the random forest on the future (by more than 0.3)",
          "the forest with t as a feature cannot extrapolate the trend: the seasonal baseline requires no model")
done()

## What you learned

- With a random K-fold the random forest looks almost perfect (R² about 0.97): for every validation month it has the nearby months in the
  training set and just interpolates the trend. On the true future its R² is much lower: it does not know what the random walk will do.
- `TimeSeriesSplit` (always trains on the past) gives an estimate much closer to the truth: the splitter is the one that imitates
  the real use of the model.
- A model that does not beat a trivial baseline (here "same month of the previous year") has no value, even with a high R² in
  cross-validation. Comparing with a baseline is the first thing to do, and it is free.

## Common pitfall

Adding `t` (or the date) as a feature to a tree model and validating randomly: the tree learns to "remember" the periods
already seen. In production `t` takes never-seen values and the model has nothing left to remember: a tree does not extrapolate.

## Interview question

> *Click on a question to show its answer.*

<details><summary>Your sales forecasting model has R² 0.95 in cross-validation but in production it is very wrong. Where do you start?</summary>

From how I validated: if I used a random K-fold on temporal data, the validation is an interpolation, not a forecast of the
future. I redo the evaluation with `TimeSeriesSplit` (or a chronological hold-out), compare with naive baselines (last value, same
period of the previous year) and check that no feature contains information from the future. If the model does not beat the baseline,
it is simply not ready.

</details>